# 实验四：同样的比特预算，谁更清晰？
## JPEG vs Learned Image Compression · Rate–Distortion Challenge

**课程**：未来媒体互联网（Future Media & Internet） &nbsp;|&nbsp; **预计时长**：~12–15 分钟  
**运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **无需 GPU** &nbsp;|&nbsp; **Internet Off**

---

### 一句话问题

> **压缩算法真正比的不是“谁的图片最小”或“谁的图片最清晰”，而是：在相同比特预算下，谁能保留更多视觉质量？**

Demo3 已经学习了 **PSNR / SSIM 如何评价质量**。  
Demo4 接着回答：**在有限传输与存储预算下，怎样权衡 Rate 与 Distortion？**


## 学习目标

完成本实验后，你应该能够：

1. 解释 **Rate–Distortion（率失真）** 为什么是压缩算法的核心评价框架；
2. 用真实 JPEG 编解码得到 **文件大小、bpp、PSNR、SSIM**；
3. 理解 JPEG `quality` 参数如何改变码率与重建质量；
4. 理解学习式压缩中的 **Encoder → Latent → Quantization → Entropy Coding → Decoder**；
5. 运行一个真正训练过的 Tiny Learned Codec，而不是 resize 模拟器；
6. 区分 **实际压缩 payload** 与“理论估计码率”；
7. 理解为什么一个小型课堂神经网络并不会天然超过成熟 JPEG；
8. 知道 CompressAI 与 JPEG AI 在学习式压缩研究和标准化中的位置。


## 核心概念：Rate–Distortion

压缩问题可以抽象为两个目标：

- **Rate \(R\)**：用了多少比特，越少越好；
- **Distortion \(D\)**：重建损失有多大，越小越好。

学习式压缩常写成：

\[
L = D + \lambda R
\]

课堂上只需要记住：

> **更好的 Codec = 相同 Rate 下质量更高，或相同质量下 Rate 更低。**

本实验中的 Tiny Learned Codec 为了保持可解释与 CPU 友好，训练时使用一个简单的 latent magnitude 作为 **rate proxy**；评测时则把量化后的 latent **真实序列化并用 zlib 压缩**，得到实际 payload bytes。  
这不是生产级 learned entropy model，但比只报告“估计 bpp”更透明。


## 运行环境

| 项目 | 设置 |
|---|---|
| 平台 | Kaggle Notebook |
| 计算资源 | CPU |
| GPU | 不需要 |
| Internet | Off |
| 外部数据集 | 不需要 |
| 测试图像 | `skimage.data.camera()` |
| 训练素材 | scikit-image 内置自然图像，不包含测试图 |
| 依赖 | PyTorch、NumPy、Pandas、Matplotlib、Pillow、scikit-image |

直接 **Run All** 即可。


In [1]:
import io
import zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from skimage import data, color, img_as_float32
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 2026
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

reference = img_as_float32(data.camera()).astype(np.float32)
H, W = reference.shape
RAW_BYTES = H * W  # 8-bit grayscale equivalent
RAW_BPP = 8.0

print("Environment ready")
print(f"Test image: {W}×{H} grayscale")
print(f"Raw 8-bit size: {RAW_BYTES / 1024:.1f} KiB ({RAW_BPP:.1f} bpp)")


# 第一幕：先建立“压缩预算”直觉

一张 512×512 的 8-bit 灰度图，未压缩像素数据约为：

\[
512\times512\times8 \text{ bits} = 256 \text{ KiB}
\]

问题不是“能不能压缩”，而是：

> **压到 40 KB、20 KB、10 KB 后，视觉质量分别还剩多少？**


In [2]:
plt.figure(figsize=(6, 6))
plt.imshow(reference, cmap="gray", vmin=0, vmax=1)
plt.title(f"Reference | raw ≈ {RAW_BYTES / 1024:.0f} KiB")
plt.axis("off")
plt.show()


# 第二幕：真实 JPEG —— 不再模拟

下面使用 Pillow 真正执行 JPEG 编码与解码。

每个 operating point 都有：

- JPEG `quality`;
- **真实 JPEG 文件字节数**；
- **bits per pixel (bpp)**；
- PSNR；
- SSIM。

因此这是一条真正可解释的 JPEG Rate–Distortion 曲线。


In [3]:
def jpeg_encode_decode(image, quality):
    arr = np.uint8(np.clip(image * 255.0, 0, 255))
    pil = Image.fromarray(arr, mode="L")

    buffer = io.BytesIO()
    pil.save(
        buffer,
        format="JPEG",
        quality=int(quality),
        optimize=True
    )
    payload = buffer.getvalue()

    decoded = np.array(
        Image.open(io.BytesIO(payload)),
        dtype=np.float32
    ) / 255.0

    size_bytes = len(payload)
    bpp = size_bytes * 8 / image.size
    psnr = peak_signal_noise_ratio(image, decoded, data_range=1.0)
    ssim = structural_similarity(image, decoded, data_range=1.0)

    return {
        "quality": int(quality),
        "bytes": size_bytes,
        "bpp": bpp,
        "psnr": psnr,
        "ssim": ssim,
        "image": decoded,
    }


JPEG_QUALITIES = [95, 80, 60, 40, 30, 20, 10, 5]
jpeg_points = [
    jpeg_encode_decode(reference, q)
    for q in JPEG_QUALITIES
]

jpeg_table = pd.DataFrame([
    {
        "JPEG Quality": p["quality"],
        "Size (KiB)": p["bytes"] / 1024,
        "bpp": p["bpp"],
        "PSNR (dB)": p["psnr"],
        "SSIM": p["ssim"],
    }
    for p in jpeg_points
])

display(
    jpeg_table.style.format({
        "Size (KiB)": "{:.2f}",
        "bpp": "{:.3f}",
        "PSNR (dB)": "{:.2f}",
        "SSIM": "{:.3f}",
    })
)


In [4]:
# Visual JPEG quality ladder
show_q = [80, 40, 20, 10]
lookup_jpeg = {p["quality"]: p for p in jpeg_points}

fig, axes = plt.subplots(1, len(show_q) + 1, figsize=(16, 4))

axes[0].imshow(reference, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Reference")
axes[0].axis("off")

for ax, q in zip(axes[1:], show_q):
    p = lookup_jpeg[q]
    ax.imshow(p["image"], cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"JPEG Q={q}\n"
        f"{p['bytes']/1024:.1f} KiB | {p['bpp']:.2f} bpp\n"
        f"PSNR {p['psnr']:.1f} | SSIM {p['ssim']:.3f}"
    )
    ax.axis("off")

plt.suptitle("Real JPEG Compression: Size ↓  →  Distortion ↑")
plt.tight_layout()
plt.show()

# Zoomed crop for blocking/ringing at low quality
roi = np.s_[120:300, 220:400]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(reference[roi], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Reference crop")
axes[0].axis("off")

axes[1].imshow(lookup_jpeg[10]["image"][roi], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("JPEG Q=10 crop")
axes[1].axis("off")

plt.suptitle("Low-quality JPEG: local artifact inspection")
plt.tight_layout()
plt.show()


In [5]:
plt.figure(figsize=(7, 5))
plt.plot(
    jpeg_table["bpp"],
    jpeg_table["PSNR (dB)"],
    marker="o",
    label="JPEG"
)
plt.xlabel("Rate (bits per pixel)")
plt.ylabel("PSNR (dB)")
plt.title("JPEG Rate–Distortion Curve")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(
    jpeg_table["bpp"],
    jpeg_table["SSIM"],
    marker="o",
    label="JPEG"
)
plt.xlabel("Rate (bits per pixel)")
plt.ylabel("SSIM")
plt.title("JPEG Rate–Quality Curve")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


# 第三幕：机器能不能自己学习一种压缩变换？

学习式图像压缩把固定变换替换成可训练网络：

**Image**  
↓  
**Encoder**  
↓  
**Latent \(z\)**  
↓  
**Quantization**  
↓  
**Entropy Coding / Bitstream**  
↓  
**Decoder**  
↓  
**Reconstruction**

这里训练一个非常小的全卷积 Autoencoder：

- 训练图像全部来自 scikit-image 内置素材；
- **测试图 `camera()` 不参与训练**；
- 从自然图像随机裁剪 32×32 patches；
- CPU 训练；
- 训练时对 latent 加均匀噪声近似量化；
- 加入一个很小的 rate proxy，鼓励 latent 更易压缩。

它是一个**真正训练过的 Toy Learned Codec**，但不是 CompressAI，也不代表生产级神经压缩性能。


In [6]:
# Build an offline patch dataset from built-in images.
# The test image camera() is deliberately excluded.

train_images = []

for fn in [data.coins, data.moon, data.page]:
    train_images.append(
        img_as_float32(fn()).astype(np.float32)
    )

for fn in [data.astronaut, data.coffee, data.chelsea, data.rocket]:
    rgb = img_as_float32(fn())
    gray = color.rgb2gray(rgb).astype(np.float32)
    train_images.append(gray)

PATCH = 32
N_PATCHES = 2200
patches = []

for _ in range(N_PATCHES):
    img = train_images[int(rng.integers(len(train_images)))]
    h, w = img.shape

    y = int(rng.integers(0, h - PATCH + 1))
    x = int(rng.integers(0, w - PATCH + 1))

    patches.append(img[y:y + PATCH, x:x + PATCH])

patch_tensor = torch.from_numpy(
    np.stack(patches)[:, None]
).float()

print(f"Training patches: {patch_tensor.shape}")


In [7]:
class TinyLearnedCodec(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(32, 8, 5, stride=2, padding=2),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x, noise_step=0.08):
        z = self.encoder(x)

        if noise_step > 0:
            # Uniform noise approximates quantization during training.
            z_for_decode = z + (
                torch.rand_like(z) - 0.5
            ) * noise_step
        else:
            z_for_decode = z

        reconstruction = self.decoder(z_for_decode)
        return reconstruction, z


model = TinyLearnedCodec()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-3
)

loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(patch_tensor),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

EPOCHS = 20
RATE_PROXY_WEIGHT = 2e-4
training_history = []

model.train()

for epoch in range(EPOCHS):
    running = 0.0

    for (batch,) in loader:
        optimizer.zero_grad()

        reconstruction, z = model(
            batch,
            noise_step=0.08
        )

        distortion = F.mse_loss(
            reconstruction,
            batch
        )

        # Classroom rate proxy:
        # smaller latent magnitude tends to be easier to quantize/compress.
        rate_proxy = torch.mean(
            torch.log1p(torch.abs(z))
        )

        loss = (
            distortion
            + RATE_PROXY_WEIGHT * rate_proxy
        )

        loss.backward()
        optimizer.step()

        running += loss.item() * len(batch)

    training_history.append(
        running / len(patch_tensor)
    )

print(
    f"Training complete: "
    f"{EPOCHS} epochs | "
    f"final loss={training_history[-1]:.5f}"
)

plt.figure(figsize=(8, 4))
plt.plot(
    np.arange(1, EPOCHS + 1),
    training_history,
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Training objective")
plt.title("Tiny Learned Codec Training Curve")
plt.grid(alpha=0.25)
plt.show()


# 第四幕：让 Learned Codec 真正产生 payload

只画 Autoencoder reconstruction 还不叫“压缩”。

这里执行完整的教学版链路：

1. Encoder 生成 latent；
2. 用不同 `quantization_step` 量化；
3. 量化整数转为字节；
4. 用 **zlib** 做无损熵压缩；
5. 用实际压缩后的 payload bytes 计算 bpp；
6. Decoder 用反量化 latent 重建图像。

因此 Learned Codec 的横轴也来自**实际字节数**。

需要强调：

> zlib 只是透明、通用的课堂熵编码器；真实学习式压缩通常使用专门训练的概率模型 + arithmetic / ANS 类熵编码，而不是直接 zlib。


In [8]:
model.eval()

reference_tensor = torch.from_numpy(
    reference
)[None, None].float()

@torch.no_grad()
def learned_encode_decode(quantization_step):
    z = model.encoder(reference_tensor)

    # Quantize latent into integer symbols.
    symbols = torch.round(
        z / quantization_step
    ).to(torch.int16)

    # Actual byte payload after lossless compression of the symbols.
    raw_latent_bytes = symbols.cpu().numpy().tobytes()
    payload = zlib.compress(
        raw_latent_bytes,
        level=9
    )

    # Decoder receives dequantized latent.
    z_quantized = symbols.float() * quantization_step
    reconstruction = model.decoder(
        z_quantized
    ).squeeze().cpu().numpy()

    size_bytes = len(payload)
    bpp = size_bytes * 8 / reference.size

    psnr = peak_signal_noise_ratio(
        reference,
        reconstruction,
        data_range=1.0
    )

    ssim = structural_similarity(
        reference,
        reconstruction,
        data_range=1.0
    )

    return {
        "qstep": float(quantization_step),
        "bytes": size_bytes,
        "bpp": bpp,
        "psnr": psnr,
        "ssim": ssim,
        "image": reconstruction,
        "latent_shape": tuple(symbols.shape),
    }


Q_STEPS = [0.05, 0.10, 0.20, 0.30, 0.50, 1.00, 2.00]
learned_points = [
    learned_encode_decode(q)
    for q in Q_STEPS
]

learned_table = pd.DataFrame([
    {
        "Quant Step": p["qstep"],
        "Payload (KiB)": p["bytes"] / 1024,
        "bpp": p["bpp"],
        "PSNR (dB)": p["psnr"],
        "SSIM": p["ssim"],
    }
    for p in learned_points
])

display(
    learned_table.style.format({
        "Quant Step": "{:.2f}",
        "Payload (KiB)": "{:.2f}",
        "bpp": "{:.3f}",
        "PSNR (dB)": "{:.2f}",
        "SSIM": "{:.3f}",
    })
)

print(
    "Latent tensor shape:",
    learned_points[0]["latent_shape"]
)


In [9]:
# Compare Rate–Distortion curves.

plt.figure(figsize=(8, 5))

plt.plot(
    jpeg_table["bpp"],
    jpeg_table["PSNR (dB)"],
    marker="o",
    label="JPEG (actual JPEG file)"
)

plt.plot(
    learned_table["bpp"],
    learned_table["PSNR (dB)"],
    marker="o",
    label="Tiny Learned Codec (zlib latent payload)"
)

plt.xlabel("Rate (bits per pixel)")
plt.ylabel("PSNR (dB)")
plt.title("Rate–Distortion: JPEG vs Tiny Learned Codec")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    jpeg_table["bpp"],
    jpeg_table["SSIM"],
    marker="o",
    label="JPEG (actual JPEG file)"
)

plt.plot(
    learned_table["bpp"],
    learned_table["SSIM"],
    marker="o",
    label="Tiny Learned Codec (zlib latent payload)"
)

plt.xlabel("Rate (bits per pixel)")
plt.ylabel("SSIM")
plt.title("Rate–Quality: JPEG vs Tiny Learned Codec")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


# 第五幕：Same Rate Challenge

现在不比较“JPEG Quality=多少”或“Neural Quantization Step=多少”。

程序自动寻找：

> **JPEG 与 Tiny Learned Codec 中 bpp 最接近的一对 operating points。**

这才是公平问题：

> **当两者实际 payload 几乎一样大时，谁的重建质量更高？**

这里不预设 Neural 必须赢。  
**实验结果是什么，就解释什么。**


In [10]:
# Find the closest-rate pair automatically.

best_pair = None
best_gap = float("inf")

for jp in jpeg_points:
    for lp in learned_points:
        gap = abs(jp["bpp"] - lp["bpp"])

        if gap < best_gap:
            best_gap = gap
            best_pair = (jp, lp)

jpeg_match, learned_match = best_pair

same_rate_table = pd.DataFrame([
    {
        "Codec": f"JPEG Q={jpeg_match['quality']}",
        "Payload (KiB)": jpeg_match["bytes"] / 1024,
        "bpp": jpeg_match["bpp"],
        "PSNR (dB)": jpeg_match["psnr"],
        "SSIM": jpeg_match["ssim"],
    },
    {
        "Codec": f"Tiny Learned q={learned_match['qstep']:.2f}",
        "Payload (KiB)": learned_match["bytes"] / 1024,
        "bpp": learned_match["bpp"],
        "PSNR (dB)": learned_match["psnr"],
        "SSIM": learned_match["ssim"],
    },
])

display(
    same_rate_table.style.format({
        "Payload (KiB)": "{:.2f}",
        "bpp": "{:.3f}",
        "PSNR (dB)": "{:.2f}",
        "SSIM": "{:.3f}",
    })
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(reference, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Reference")
axes[0].axis("off")

axes[1].imshow(
    jpeg_match["image"],
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[1].set_title(
    f"JPEG Q={jpeg_match['quality']}\n"
    f"{jpeg_match['bpp']:.3f} bpp\n"
    f"PSNR {jpeg_match['psnr']:.2f} | SSIM {jpeg_match['ssim']:.3f}"
)
axes[1].axis("off")

axes[2].imshow(
    learned_match["image"],
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[2].set_title(
    f"Tiny Learned q={learned_match['qstep']:.2f}\n"
    f"{learned_match['bpp']:.3f} bpp\n"
    f"PSNR {learned_match['psnr']:.2f} | SSIM {learned_match['ssim']:.3f}"
)
axes[2].axis("off")

plt.suptitle(
    f"Same Rate Challenge | rate gap = {best_gap:.4f} bpp"
)
plt.tight_layout()
plt.show()

winner = (
    "JPEG"
    if jpeg_match["psnr"] > learned_match["psnr"]
    else "Tiny Learned Codec"
)

print(f"Higher PSNR at this matched rate: {winner}")


# 结果应该怎样解读？

本实验最重要的不是“Neural 必须赢”。

如果运行结果显示 **JPEG 优于 Tiny Learned Codec**，这是完全合理的：

- JPEG 是成熟标准，工程实现经过多年优化；
- Toy Learned Codec 只有很小的网络和极少训练数据；
- 它没有 hyperprior / context model；
- 它没有真正学习概率分布的 entropy model；
- zlib 并不是为 learned latent 专门设计的熵编码器；
- 我们只训练了一个统一小模型，并通过 quantization step 扫描 operating points。

因此正确结论是：

> **“Learning-based compression”描述的是一种可学习的端到端率失真优化范式，而不是“任何神经网络都自动优于传统 Codec”。**

真正的研究问题是：

> **怎样设计更好的变换、概率模型、量化与训练目标，让整条 RD Curve 向左上移动？**


# 从课堂 Toy Codec 到真实学习式压缩

可以把技术层次分成三层：

### 1. 本实验：Toy Learned Codec
用于理解：

**Encoder → Latent → Quantization → Payload → Decoder → Rate–Distortion**

### 2. 研究工具：CompressAI
CompressAI 是 InterDigital AI Lab 维护的 **PyTorch end-to-end compression research library and evaluation platform**，提供预训练 learned image compression 模型，以及与传统 image/video codecs 的评测工具。

https://github.com/InterDigitalInc/CompressAI

### 3. 标准化：JPEG AI
JPEG AI（ISO/IEC 6048 / ITU-T T.840）是首个基于端到端学习方法的国际图像编码标准。Version 1 面向人类视觉消费，于 2025 年发布。

https://jpeg.org/jpegai/

因此，学习式图像压缩已经从：

**Research → Tooling → International Standard**

逐步进入现实技术体系。


## 实验局限性与思考

### 本实验有意做了哪些简化？

1. 只处理灰度图像；
2. 训练数据来自少量内置自然图像 patch；
3. Tiny Codec 没有 hyperprior / context model；
4. 训练中的 `rate_proxy` 不是真实熵模型；
5. zlib 只是课堂用透明熵编码器；
6. 没有把模型权重计入每张图片 payload——和传统 codec 一样，默认编解码器已预部署；
7. 没有训练多个不同 \(\lambda\) 的独立 RD 模型；
8. 测试只有一张图，不能代表数据集级 benchmark。

### 思考题

1. 为什么在同样 bpp 下，成熟 JPEG 可能明显优于这个 Toy Neural Codec？
2. 如果把训练 patch 增加 100 倍，结果一定会超过 JPEG 吗？
3. Quantization Step 增大时，Rate 和 Distortion 为什么会同时变化？
4. 如果用 SSIM 代替 MSE 训练 Decoder，RD 曲线可能怎样变化？
5. Hyperprior 的核心作用为什么与“更准确地预测 latent 概率”有关？
6. Demo3 的 PSNR / SSIM 如何成为 Demo4 的 Distortion 指标？
7. 如果把任务扩展到视频，除了空间压缩，还需要利用什么时间冗余？


---

← [实验三：视频质量评估 PSNR vs SSIM](https://www.kaggle.com/code/guopingtan/fmi-demo3-quality-assessment)
&nbsp;|&nbsp;
🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here)
&nbsp;|&nbsp;
[实验五：语义通信 vs 传统传输 →](https://www.kaggle.com/code/guopingtan/fmi-demo5-semantic-communication)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University
